In [ ]:
import os
import sys
import plotly.express as px
import logging
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Javascript
sys.path.append("../../../..")
sys.path.append("../../../../scripts")
sys.path.append("../../../../scripts/summarize/calibration")

from notebooks.notebook_styling import bellevue_theme
from input_configuration import *
from h5toDF import *
from summary_functions import *
from dictionary import *
from utils import survey_year, get_subarea, get_data

logging.disable(logging.CRITICAL)

In [ ]:
# data processing
taz_subarea = pd.read_csv(os.path.join(project_folder, districtfile))
taz_subarea['DistrictFlowName'] = taz_subarea['DistrictFlowID'].map(district_flow_name)
taz_subarea.rename(columns={'BKRCastTAZ': 'TAZ'}, inplace=True)
data_daysim = convert(os.path.join(project_folder, h5_results_file), 
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_results_name), stdout=False)
data_survey = convert(os.path.join(project_folder, h5_comparison_file),   # data_survey and data_fullsurvey are different in 2023, data_survey has a smaller number of records
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_comparison_name), stdout=False)
data_fullsurvey = convert(os.path.join(project_folder, h5_fullsurvey_file), 
                          os.path.join(project_folder, guidefile), 
                          os.path.join(project_folder, h5_fullsurvey_name), stdout=False)

data_survey['Trip_cloned']['mode'] = data_survey['Trip_cloned']['mode'].replace('TNC','Other')
data_fullsurvey['Trip']['mode'] = data_fullsurvey['Trip']['mode'].replace('TNC','Other')

# locate the ACS survey data
acs_data = os.path.join(project_folder, f'inputs/model/survey/ACS_2023.xlsx')
acs_data_bkr = os.path.join(project_folder, f'inputs/model/survey/ACS_2023_BKR.xlsx')

## Total Trips

In [ ]:
def total_trips(data1, data2, tag='PSRC Region'):
    Trip_1_total = get_total(data1['Trip']['trexpfac'])
    if int(model_year) > 2023:
        Trip_2_total = get_total(data2['Trip']['trexpfac'])
    else:
        Trip_2_total = get_total(data2['Trip_cloned']['trexpfac'])

    tpp  = pd.DataFrame(index = ['Trip'])
    tpp['DaysimOutputs'] = Trip_1_total
    tpp[f'{survey_year}Survey'] = Trip_2_total
    tpp = get_differences(tpp, 'DaysimOutputs', f'{survey_year}Survey', 2)
    # table
    display(tpp.style.format({
        'DaysimOutputs': '{:,.0f}',
        f'{survey_year}Survey': '{:,.0f}',
        f"Difference (DaysimOutputs - {survey_year}Survey)": '{:,.0f}',
        f"% Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}%',}))

In [ ]:
if int(model_year) > 2023:
    total_trips(data1=data_daysim, data2=data_fullsurvey, tag='PSRC Region')
else:
    total_trips(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
import copy
_data_daysim = copy.deepcopy(data_daysim)
_data_survey = copy.deepcopy(data_survey)
_data_fullsurvey = copy.deepcopy(data_fullsurvey)
fname_tail, data_daysim_bkr, data_survey_bkr, data_fullsurvey_bkr = \
    get_data(data1=_data_daysim, data2=_data_survey, data3=_data_fullsurvey, taz_subarea=taz_subarea, if_region=False)

if int(model_year) > 2023:
    total_trips(data1=data_daysim_bkr, data2=data_fullsurvey_bkr, tag='BKR')
else:
    total_trips(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Trips per Person

In [ ]:
def trip_per_person(data1, data2, tag='PSRC Region'):
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    Trip_1_total = get_total(data1['Trip']['trexpfac'])
    Trip_2_total = get_total(data2['Trip']['trexpfac'])

    ##Trips per person
    tpp1 = Trip_1_total / Person_1_total
    tpp2 = Trip_2_total / Person_2_total
    tpp  = pd.DataFrame(index = ['Trip'])
    tpp['DaysimOutputs'] = tpp1
    tpp[f'{survey_year}Survey'] = tpp2
    tpp = get_differences(tpp, 'DaysimOutputs', f'{survey_year}Survey', 2)
    # table
    display(tpp.style.format({
        'DaysimOutputs': '{:,.2f}',
        f'{survey_year}Survey': '{:,.2f}',
        f"Difference (DaysimOutputs - {survey_year}Survey)": '{:,.2f}',
        f"% Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}%',}))

In [ ]:
trip_per_person(data1=data_daysim, data2=data_fullsurvey, tag='PSRC Region')

In [ ]:
trip_per_person(data1=data_daysim_bkr, data2=data_fullsurvey_bkr, tag='BKR')

## Trips per Person by Purpose

In [ ]:
def trips_per_ps_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Trips per Person by Purpose
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    tpbp1 = data1['Trip'][['dpurp','trexpfac']].groupby('dpurp').sum()['trexpfac'] / Person_1_total
    tpbp2 = data2['Trip_cloned'][['dpurp','trexpfac']].groupby('dpurp').sum()['trexpfac'] / Person_2_total
    tpbp = pd.DataFrame()
    tpbp['Trips per Person (DaysimOutputs)'] = tpbp1
    tpbp[f'Trips per Person ({survey_year}Survey)'] = tpbp2
    tpbp = get_differences(tpbp, 'Trips per Person (DaysimOutputs)', f'Trips per Person ({survey_year}Survey)', 2)
    tpbp = recode_index(tpbp, 'dpurp', 'Trip Purpose')
    tpbp = tpbp.loc[pdpurp_cat.values()]
    # table
    display(tpbp.style.format({
        'Trips per Person (DaysimOutputs)': '{:,.2f}',
        f'Trips per Person ({survey_year}Survey)': '{:,.2f}',
        f"Difference (Trips per Person (DaysimOutputs) - Trips per Person ({survey_year}Survey))": '{:,.2f}',
        f"% Difference (Trips per Person (DaysimOutputs) - Trips per Person ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        tpbp.reset_index(),
        x='Trip Purpose',
        y=['Trips per Person (DaysimOutputs)', f'Trips per Person ({survey_year}Survey)'],
        barmode='group',
        title=f'Trips per Person by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Trips per Person', xaxis_title='Trip Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()    

In [ ]:
trips_per_ps_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
trips_per_ps_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Trips per Person by Mode

In [ ]:
def trips_per_ps_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Trips per Person by Mode
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    tpbp1 = data1['Trip'][['mode','trexpfac']].groupby('mode').sum()['trexpfac'] / Person_1_total
    tpbp2 = data2['Trip_cloned'][['mode','trexpfac']].groupby('mode').sum()['trexpfac'] / Person_2_total
    tpbp = pd.DataFrame()
    tpbp['Trips per Person (DaysimOutputs)'] = tpbp1
    tpbp[f'Trips per Person ({survey_year}Survey)'] = tpbp2
    tpbp = get_differences(tpbp, 'Trips per Person (DaysimOutputs)', f'Trips per Person ({survey_year}Survey)', 2)
    tpbp = recode_index(tpbp, 'mode', 'Trip Mode')
    tpbp = tpbp.loc[trip_mode_cat.values()]
    # table
    display(tpbp.style.format({
        'Trips per Person (DaysimOutputs)': '{:,.2f}',
        f'Trips per Person ({survey_year}Survey)': '{:,.2f}',
        f"Difference (Trips per Person (DaysimOutputs) - Trips per Person ({survey_year}Survey))": '{:,.2f}',
        f"% Difference (Trips per Person (DaysimOutputs) - Trips per Person ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        tpbp.reset_index(),
        x='Trip Mode',
        y=['Trips per Person (DaysimOutputs)', f'Trips per Person ({survey_year}Survey)'],
        barmode='group',
        title=f'Trips per Person by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Trips per Person', xaxis_title='Trip Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()    

In [ ]:
trips_per_ps_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
trips_per_ps_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Trip Share by Purpose

In [ ]:
def pc_trip_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Percent of Trips by Purpose
    trip_1_total = get_total(data1['Trip']['trexpfac'])
    trip_2_total = get_total(data2['Trip_cloned']['trexpfac'])
    ptbp1 = 100 * data1['Trip'][['dpurp','trexpfac']].groupby('dpurp').sum()['trexpfac'] / trip_1_total
    ptbp2 = 100 * data2['Trip_cloned'][['dpurp','trexpfac']].groupby('dpurp').sum()['trexpfac'] / trip_2_total
    ptbp = pd.DataFrame()
    ptbp['Percent of Trips (DaysimOutputs)'] = ptbp1
    ptbp[f'Percent of Trips ({survey_year}Survey)'] = ptbp2
    ptbp = get_differences(ptbp,'Percent of Trips (DaysimOutputs)', f'Percent of Trips ({survey_year}Survey)', 2)
    ptbp = recode_index(ptbp, 'dpurp', 'Trips Purpose')
    ptbp = ptbp.loc[pdpurp_cat.values()]
    # table
    display(ptbp.style.format({
        'Percent of Trips (DaysimOutputs)': '{:,.1f}%',
        f'Percent of Trips ({survey_year}Survey)': '{:,.1f}%',
        f"Difference (Percent of Trips (DaysimOutputs) - Percent of Trips ({survey_year}Survey))": '{:,.1f}%',
        f"% Difference (Percent of Trips (DaysimOutputs) - Percent of Trips ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        ptbp.reset_index(),
        x='Trips Purpose',
        y=['Percent of Trips (DaysimOutputs)', f'Percent of Trips ({survey_year}Survey)'],
        barmode='group',
        title=f'Percent of Trips by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Percent of Trips', xaxis_title='Trips Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.update_yaxes(ticksuffix='%')
    fig.show()

In [ ]:
pc_trip_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
pc_trip_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Trip Share by Mode

In [ ]:
def pc_trip_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Percent of Trips by Mode
    trip_1_total = get_total(data1['Trip']['trexpfac'])
    trip_2_total = get_total(data2['Trip_cloned']['trexpfac'])
    ptbp1 = 100 * data1['Trip'][['mode','trexpfac']].groupby('mode').sum()['trexpfac'] / trip_1_total
    ptbp2 = 100 * data2['Trip_cloned'][['mode','trexpfac']].groupby('mode').sum()['trexpfac'] / trip_2_total
    ptbp = pd.DataFrame()
    ptbp['Percent of Trips (DaysimOutputs)'] = ptbp1
    ptbp[f'Percent of Trips ({survey_year}Survey)'] = ptbp2
    ptbp = get_differences(ptbp,'Percent of Trips (DaysimOutputs)', f'Percent of Trips ({survey_year}Survey)', 2)
    ptbp = recode_index(ptbp, 'mode', 'Trips Mode')
    ptbp = ptbp.loc[trip_mode_cat.values()]
    # table
    display(ptbp.style.format({
        'Percent of Trips (DaysimOutputs)': '{:,.1f}%',
        f'Percent of Trips ({survey_year}Survey)': '{:,.1f}%',
        f"Difference (Percent of Trips (DaysimOutputs) - Percent of Trips ({survey_year}Survey))": '{:,.1f}%',
        f"% Difference (Percent of Trips (DaysimOutputs) - Percent of Trips ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        ptbp.reset_index(),
        x='Trips Mode',
        y=['Percent of Trips (DaysimOutputs)', f'Percent of Trips ({survey_year}Survey)'],
        barmode='group',
        title=f'Percent of Trips by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Percent of Trips', xaxis_title='Trips Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.update_yaxes(ticksuffix='%')
    fig.show()

In [ ]:
pc_trip_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
pc_trip_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Trip Distance by Purpose

In [ ]:
def trips_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(trip_ok_1[['travdist', 'trexpfac', 'dpurp']], 'travdist', 'trexpfac', 'dpurp')
    triptotal2 = weighted_average(trip_ok_2[['travdist', 'trexpfac', 'dpurp']], 'travdist', 'trexpfac', 'dpurp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Trip Length (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Trip Length (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Trip Length (' + name1 + ')', 'Average Trip Length (' + name2 + ')', 2)
    atl = recode_index(atl, 'dpurp', 'Trip Purpose')  
    atl = atl.loc[pdpurp_cat.values()]
    # display table
    display(atl.style.format({
        f'Average Trip Length ({name1})': '{:,.2f}',
        f'Average Trip Length ({name2})': '{:,.2f}',
        f'Difference (Average Trip Length ({name1}) - Average Trip Length ({name2}))': '{:,.2f}',
        f'% Difference (Average Trip Length ({name1}) - Average Trip Length ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Trip Purpose',
        y=[f'Average Trip Length ({name1})', f'Average Trip Length ({name2})'],
        barmode='group',
        title=f'Average Trip Distance by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Average Trip Distance', xaxis_title='Trip Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
trips_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
trips_distance_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Trip Distance by Mode

In [ ]:
def trips_distance_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour mode
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)

    #Compute weighted average of trip length grouped by mode
    triptotal1 = weighted_average(trip_ok_1[['travdist', 'trexpfac', 'mode']], 'travdist', 'trexpfac', 'mode')
    triptotal2 = weighted_average(trip_ok_2[['travdist', 'trexpfac', 'mode']], 'travdist', 'trexpfac', 'mode')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Trip Length (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Trip Length (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Trip Length (' + name1 + ')', 'Average Trip Length (' + name2 + ')', 2)
    atl = recode_index(atl, 'mode', 'Trip Mode')  
    atl = atl.loc[trip_mode_cat.values()]
    # display table
    display(atl.style.format({
        f'Average Trip Length ({name1})': '{:,.2f}',
        f'Average Trip Length ({name2})': '{:,.2f}',
        f'Difference (Average Trip Length ({name1}) - Average Trip Length ({name2}))': '{:,.2f}',
        f'% Difference (Average Trip Length ({name1}) - Average Trip Length ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Trip Mode',
        y=[f'Average Trip Length ({name1})', f'Average Trip Length ({name2})'],
        barmode='group',
        title=f'Average Trip Distance by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Average Trip Distance', xaxis_title='Trip Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
trips_distance_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
trips_distance_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Trip Travel Time by Purpose

In [ ]:
def trips_tt_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by trip purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(trip_ok_1[['travtime', 'trexpfac', 'dpurp']], 'travtime', 'trexpfac', 'dpurp')
    triptotal2 = weighted_average(trip_ok_2[['travtime', 'trexpfac', 'dpurp']], 'travtime', 'trexpfac', 'dpurp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Trip Travel Time (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Trip Travel Time (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Trip Travel Time (' + name1 + ')', 'Average Trip Travel Time (' + name2 + ')', 2)
    atl = recode_index(atl, 'dpurp', 'Trip Purpose')  
    atl.columns.name = 'Travel Time (minutes)'
    atl = atl.loc[pdpurp_cat.values(), :]
    # display table
    display(atl.style.format({
        f'Average Trip Travel Time ({name1})': '{:,.1f}',
        f'Average Trip Travel Time ({name2})': '{:,.1f}',
        f'Difference (Average Trip Travel Time ({name1}) - Average Trip Travel Time ({name2}))': '{:,.1f}',
        f'% Difference (Average Trip Travel Time ({name1}) - Average Trip Travel Time ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Trip Purpose',
        y=[f'Average Trip Travel Time ({name1})', f'Average Trip Travel Time ({name2})'],
        barmode='group',
        title=f'Average Trip Travel Time by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Average Trip Travel Time', xaxis_title='Trip Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
trips_tt_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
trips_tt_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Trip Travel Time by Mode

In [ ]:
def trips_tt_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by trip mode
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)

    #Compute weighted average of trip length grouped by mode
    triptotal1 = weighted_average(trip_ok_1[['travtime', 'trexpfac', 'mode']], 'travtime', 'trexpfac', 'mode')
    triptotal2 = weighted_average(trip_ok_2[['travtime', 'trexpfac', 'mode']], 'travtime', 'trexpfac', 'mode')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Trip Travel Time (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Trip Travel Time (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Trip Travel Time (' + name1 + ')', 'Average Trip Travel Time (' + name2 + ')', 2)
    atl = recode_index(atl, 'mode', 'Trip Mode')  
    atl.columns.name = 'Travel Time (minutes)'
    atl = atl.loc[trip_mode_cat.values(), :]
    # display table
    display(atl.style.format({
        f'Average Trip Travel Time ({name1})': '{:,.1f}',
        f'Average Trip Travel Time ({name2})': '{:,.1f}',
        f'Difference (Average Trip Travel Time ({name1}) - Average Trip Travel Time ({name2}))': '{:,.1f}',
        f'% Difference (Average Trip Travel Time ({name1}) - Average Trip Travel Time ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Trip Mode',
        y=[f'Average Trip Travel Time ({name1})', f'Average Trip Travel Time ({name2})'],
        barmode='group',
        title=f'Average Trip Travel Time by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Average Trip Travel Time', xaxis_title='Trip Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [ ]:
trips_tt_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

In [ ]:
trips_tt_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

## Trips by District

In [ ]:
##Trips per Person by Purpose and Person Type/Number of Stops
data1=data_daysim
data2=data_survey
name1 = 'DaysimOutputs'
name2 = f'{survey_year}Survey' 
# calculate the percentage of each number of stops by purpose
data1['Household'] = data1['Household'].merge(taz_subarea[['TAZ', 'DistrictFlowName']], left_on='hhtaz', right_on='TAZ', how='left')
data1['Trip'] = data1['Trip'].merge(data1['Household'][['hhno', 'DistrictFlowName', 'hhincome', 'hhvehs']], on='hhno', how='left')
data2['Household'] = data2['Household'].merge(taz_subarea[['TAZ', 'DistrictFlowName']], left_on='hhtaz', right_on='TAZ', how='left')
data2['Trip_cloned'] = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'DistrictFlowName', 'hhincome', 'hhvehs']], on='hhno', how='left')
todist1 = data1['Trip'].groupby(by='DistrictFlowName')['trexpfac'].sum()
todist2 = data2['Trip_cloned'].groupby(by='DistrictFlowName')['trexpfac'].sum()
df_compare = pd.concat([todist1, todist2], axis=1)
df_compare.columns = [name1, name2]
df_compare['Difference'] = df_compare[name1] - df_compare[name2]
df_compare['% Difference'] = (df_compare['Difference'] / df_compare[name2]) * 100
display(df_compare.loc[district_flow_name.values()].style.format({
    name1: '{:,.0f}',
    name2: '{:,.0f}',
    'Difference': '{:,.0f}',
    '% Difference': '{:,.1f}%'
}))

In [ ]:
def trips_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='DistrictFlowName', gp2='pdpurp', 
                        gp1_label='DistrictFlowName', gp2_label='Trip Purpose',
                        gp1_list=[], gp2_list=[]):
    trip_by_district_purpose1 = data1['Trip'].groupby([gp1, gp2], observed=False)['trexpfac'].sum().unstack(gp2)
    trip_by_district_purpose2 = data2['Trip_cloned'].groupby([gp1, gp2], observed=False)['trexpfac'].sum().unstack(gp2)
    # Reorder index and columns as requested
    trip_by_district_purpose1 = trip_by_district_purpose1.loc[list(gp1_list), list(gp2_list)]
    trip_by_district_purpose2 = trip_by_district_purpose2.loc[list(gp1_list), list(gp2_list)]
    trip_by_district_purpose1.columns.name = gp2_label
    trip_by_district_purpose2.columns.name = gp2_label
    trip_by_district_purpose1.index.name = gp1_label
    trip_by_district_purpose2.index.name = gp1_label
    display(trip_by_district_purpose1.style.format('{:,.0f}').set_caption("DaysimOutputs"))
    display(trip_by_district_purpose2.style.format('{:,.0f}').set_caption(f"{survey_year}Survey"))
    percent_diff = (trip_by_district_purpose1 - trip_by_district_purpose2) / trip_by_district_purpose2 * 100
    display(percent_diff.style.format('{:,.1f}%').set_caption("Percentage Difference (DaysimOutputs - Survey)"))

In [ ]:
def trip_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='DistrictFlowName', gp2='pdpurp', 
                        gp1_label='DistrictFlowName', gp2_label='Trip Purpose',
                        gp1_list=[], gp2_list=[]):
    trip_by_district_purpose1 = data1['Trip'].groupby([gp1, gp2], observed=False)['trexpfac'].sum().unstack(gp2)
    trip_by_district_purpose2 = data2['Trip_cloned'].groupby([gp1, gp2], observed=False)['trexpfac'].sum().unstack(gp2)
    # Reorder index and columns as requested
    trip_by_district_purpose1 = trip_by_district_purpose1.loc[list(gp1_list), list(gp2_list)]
    trip_by_district_purpose2 = trip_by_district_purpose2.loc[list(gp1_list), list(gp2_list)]
    trip_by_district_purpose1.columns.name = gp2_label
    trip_by_district_purpose2.columns.name = gp2_label
    trip_by_district_purpose1.index.name = gp1_label
    trip_by_district_purpose2.index.name = gp1_label
    trip_by_district_purpose1 = trip_by_district_purpose1.div(trip_by_district_purpose1.sum(axis=0), axis=1) * 100
    trip_by_district_purpose2 = trip_by_district_purpose2.div(trip_by_district_purpose2.sum(axis=0), axis=1) * 100
    def pct_fmt(x):
        return 'nan' if pd.isna(x) else f'{x:,.1f}%'
    display(trip_by_district_purpose1.style.format(pct_fmt).set_caption("DaysimOutputs"))
    display(trip_by_district_purpose2.style.format(pct_fmt).set_caption(f"{survey_year}Survey"))

## Trips by District by Purpose

In [ ]:
# :::{.panel-tabset}

# ### Number of Trips

In [ ]:
# trips_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
#                         gp1='dpurp', gp2='DistrictFlowName', 
#                         gp1_label='Trip Purpose', gp2_label='DistrictFlowName',
#                         gp1_list=pdpurp_cat.values(), gp2_list=district_flow_name.values())

In [ ]:
### Trip Share

In [ ]:
trip_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='dpurp', gp2='DistrictFlowName', 
                        gp1_label='Trip Purpose', gp2_label='DistrictFlowName',
                        gp1_list=pdpurp_cat.values(), gp2_list=district_flow_name.values())

In [ ]:
# :::

## Trips by District by Mode

In [ ]:
# :::{.panel-tabset}

# ### Number of Trips

In [ ]:
# trips_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
#                     gp1='DistrictFlowName', gp2='mode', 
#                     gp1_label='DistrictFlowName', gp2_label='Trip Mode',
#                     gp1_list=district_flow_name.values(), gp2_list=trip_mode_cat.values())

In [ ]:
### Trip Share

In [ ]:
trip_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                    gp1='DistrictFlowName', gp2='mode', 
                    gp1_label='DistrictFlowName', gp2_label='Trip Mode',
                    gp1_list=district_flow_name.values(), gp2_list=trip_mode_cat.values())

In [ ]:
# :::